# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [1]:
# Optional setup: install dependencies if they are missing in your environment.
%pip install -q transformers torch


In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "The spicy dog flew over the cuckoo's nest."
print(sample_sentence)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

The spicy dog flew over the cuckoo's nest.


In [3]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # TODO: adjust if your sentence needs more room
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)


index | token        | id
-------------------------
    0 | [CLS]        |   101
    1 | the          |  1996
    2 | spicy        | 25482
    3 | dog          |  3899
    4 | flew         |  5520
    5 | over         |  2058
    6 | the          |  1996
    7 | cuckoo       | 29010
    8 | '            |  1005
    9 | s            |  1055
   10 | nest         |  9089
   11 | .            |  1012
   12 | [SEP]        |   102
   13 | [PAD]        |     0
   14 | [PAD]        |     0
   15 | [PAD]        |     0
   16 | [PAD]        |     0
   17 | [PAD]        |     0
   18 | [PAD]        |     0
   19 | [PAD]        |     0
   20 | [PAD]        |     0
   21 | [PAD]        |     0
   22 | [PAD]        |     0
   23 | [PAD]        |     0

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Special tokens (index, token): [(0, '[CLS]'), (12, '[SEP]'), (13, '[PAD]'), (14, '[PAD]'), (15, '[PAD]'), (16, '[PAD]'), (17, '[PAD]'), (18, '[PAD]'), (19, '[PAD]

### Exercise 1 reflection
- **Describe how [CLS] and [SEP] behave inside the encoder:**
  - **`[CLS]` (Classification)**: Inserted at the beginning of every sequence. During self-attention layers, it aggregates information from all other tokens in the sentence, acting as a sentence-level representation. Its final hidden state is typically used as the input for sequence-level classification tasks.
  - **`[SEP]` (Separator)**: Placed at the end of a sentence or between two different sentences (e.g., in Next Sentence Prediction). It acts as a boundary marker, helping the model learn sequence-level splits and distinguish between multiple inputs.

- **Explain how the attention mask hides padded positions from self-attention:**
  - The attention mask is a binary tensor (with `1` for actual tokens and `0` for padded `[PAD]` tokens). During the self-attention calculation, the attention mask is used to add a very large negative value (usually $-10000.0$) to the attention logits of padded tokens before the Softmax operation. Consequently, when Softmax is applied, the attention weights for `[PAD]` tokens drop to virtually zero ($0$), ensuring the model completely ignores them and prevents them from influencing representations of the active tokens.

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [4]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "This apple pie recipe is ridiculously tasty."
prediction = sentiment_pipeline(sentence)
prediction


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9995107650756836}]

### Exercise 2 reflection
- **Does the predicted label match your expectation? Why or why not?**
  - Yes, the predicted label is **POSITIVE**, which perfectly matches expectations. The word "ridiculously" is often used to emphasize intense emotions, and combined with "tasty" in the context of food, it carries an overwhelmingly positive sentiment.

- **How confident is the model and what does the score tell you?**
  - The model is extremely confident, with a confidence score of **~99.95%** (0.99951). This score represents the softmax probability output by the model for the "POSITIVE" class, indicating that there is almost zero ambiguity or uncertainty in its prediction.

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        # Load the tokenizer and model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.max_length = max_length

        # Move the model to the proper device (GPU if available, else CPU)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()  # Set the model to evaluation mode

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        # Tokenize and return tensors on the correct device
        encoded = self.tokenizer(
            text,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {key: val.to(self.device) for key, val in encoded.items()}

    def predict(self, text: str) -> Dict[str, float]:
        # Prepare inputs
        inputs = self.preprocess(text)

        # Run inference without gradient tracking
        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits
            probabilities = torch.softmax(logits, dim=-1).squeeze(0)

        # Map outputs to labels
        # Note: distilbert-base-uncased-finetuned-sst-2-english uses indices: 0 -> NEGATIVE, 1 -> POSITIVE
        id2label = self.model.config.id2label
        pred_label_id = torch.argmax(probabilities).item()

        return {
            "label": id2label[pred_label_id],
            "confidence": probabilities[pred_label_id].item()
        }

In [6]:
# Instantiate your analyzer and test several sentences once the class is ready.
analyzer = BERTSentimentAnalyzer()
samples = [
    "I absolutely love learning about natural language processing, it is so exciting!",
    "The weather today is miserable and everything has been going wrong."
]
for text in samples:
    print(f"Text: {text}")
    print(f"Result: {analyzer.predict(text)}\n")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Text: I absolutely love learning about natural language processing, it is so exciting!
Result: {'label': 'POSITIVE', 'confidence': 0.9998655319213867}

Text: The weather today is miserable and everything has been going wrong.
Result: {'label': 'NEGATIVE', 'confidence': 0.9996299743652344}



## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

### Deliverables & Reflection
- **Detected Entities from Sample Text:**
  - `Elon Musk` (Entity: `ORG` / `PER` depending on tokenization; detected as `ORG` or `PER` by the model)
  - `SpaceX` (Entity: `ORG`)
  - `Hawthorne` (Entity: `LOC`)
  - `California` (Entity: `LOC`)

- **How we handled subword tokens starting with `##`:**
  - During tokenization, BERT uses WordPiece which splits out-of-vocabulary words into smaller, frequently-occurring subword units. Subwords that are continuations of a main word begin with a double-hash (`##`) symbol.
  - In our `recognize` method, we kept track of these continuations by checking if the current token starts with `##`. If it does, we strip the `##` prefix and append the remaining characters directly to the previous token's text without adding a space (e.g., `['Space', '##X']` is reconstructed as `SpaceX`). This allows us to cleanly stitch fractured words back together into human-readable entity spans while maintaining correct start/end offsets.

In [7]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        # Load the tokenizer and token classification model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)

        # Detect the available device (GPU or CPU)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()  # Set the model to evaluation mode

    def recognize(self, text: str):
        # Tokenize inputs and keep offsets to map tokens back to original string characters
        inputs = self.tokenizer(text, return_offsets_mapping=True, return_tensors="pt")
        offset_mapping = inputs["offset_mapping"][0].tolist()

        input_ids = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)

        # Run the forward pass
        with torch.no_grad():
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1)[0].tolist()

        tokens = self.tokenizer.convert_ids_to_tokens(input_ids[0].tolist())
        id2label = self.model.config.id2label

        entities = []
        current_entity = None

        for idx, (token, pred_id, offsets) in enumerate(zip(tokens, predictions, offset_mapping)):
            # Skip special tokens
            if token in ["[CLS]", "[SEP]", "[PAD]"]:
                continue

            label = id2label[pred_id]
            start, end = offsets

            # If the token is a subword (starts with ##)
            is_subword = token.startswith("##")
            clean_token = token[2:] if is_subword else token

            # If we are continuing an entity or processing a subword
            if label != "O":
                # Determine entity type (e.g., 'PER', 'LOC', 'ORG')
                entity_type = label.split("-")[-1]

                if current_entity and (label.startswith("I-") or is_subword) and current_entity["entity"] == entity_type:
                    # Append to current entity
                    if is_subword:
                        current_entity["text"] += clean_token
                    else:
                        # Handle spacing for normal tokens
                        current_entity["text"] += " " + clean_token
                    current_entity["end"] = end
                else:
                    # Save previous entity if exists
                    if current_entity:
                        entities.append(current_entity)
                    # Start a new entity
                    current_entity = {
                        "text": clean_token,
                        "entity": entity_type,
                        "start": start,
                        "end": end
                    }
            else:
                # If it's a subword of a current active entity, we merge it even if the subword is marked 'O'
                if current_entity and is_subword:
                    current_entity["text"] += clean_token
                    current_entity["end"] = end
                else:
                    # No active entity, or end of active entity
                    if current_entity:
                        entities.append(current_entity)
                        current_entity = None

        if current_entity:
            entities.append(current_entity)

        # Clean up any potential token artifacts in text fields
        for ent in entities:
            ent["text"] = ent["text"].strip()

        return entities

In [8]:
ner = BERTNamedEntityRecognizer()
sample_text = "On Monday, Elon Musk visited the SpaceX headquarters in Hawthorne, California."
results = ner.recognize(sample_text)
for ent in results:
    print(ent)

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'text': 'Elon Musk', 'entity': 'ORG', 'start': 11, 'end': 20}
{'text': 'SpaceX', 'entity': 'ORG', 'start': 33, 'end': 39}
{'text': 'Hawthorne', 'entity': 'LOC', 'start': 56, 'end': 65}
{'text': 'California', 'entity': 'LOC', 'start': 67, 'end': 77}


## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

Fill the table with concise statements (one line each).

| Category | BERT | GPT |
|----------|------|-----|
| **Architecture** | Bidirectional encoder (self-attention looks at past and future tokens). | Auto-regressive decoder (masked self-attention looks only at past tokens). |
| **Primary purpose** | Understanding, representation, and extraction of semantic context. | Text generation and sequence continuation. |
| **Typical use cases** | Sentiment analysis, Named Entity Recognition, Question Answering, Search. | Chatbots, summarization, creative writing, code generation. |
| **Strengths** | Rich bidirectional context; captures deep sentence semantics easily. | Outstanding generative fluency, zero-shot/few-shot in-context learning. |
| **Weaknesses** | Poor at text generation; struggles with autoregressive or seq2seq generation tasks. | Prone to hallucinations; computationally expensive to run inference sequentially. |

## Exercise 6 - BERT inside Retrieval-Augmented Generation
Objective: Explain how BERT-generated embeddings power the retrieval stage of a RAG workflow.

Address each bullet with a short paragraph:

1. **How BERT encodes queries and documents:**
   BERT processes text bi-directionally, allowing it to capture the deep contextual semantics of sentences. To prepare documents for retrieval, large texts are chunked and passed through a bi-encoder (like a Sentence-BERT model) to generate dense, fixed-dimensional vector embeddings for each chunk. When a user submits a query, it is passed through the same model, projecting both the query and the historical documents into a shared latent semantic space.

2. **How those embeddings are stored and searched in a vector database:**
   Once document chunk embeddings are generated, they are stored in a specialized vector database (such as Pinecone, Milvus, or FAISS) alongside their metadata. When a user query embedding is generated at runtime, the database performs an Approximate Nearest Neighbor (ANN) search using similarity metrics like cosine similarity or dot product. This matches the semantic intent of the query with the most semantically relevant document chunks in milliseconds.

3. **How the retrieved passages are handed to a generative model like GPT:**
   The top-$k$ most similar document chunks retrieved from the vector database are extracted as plain text. These chunks are stuffed into a structured prompt template as "context," alongside the original user query. This combined context-rich prompt is then sent to an autoregressive model (such as GPT-4), which uses the retrieved factual content to construct a coherent, accurate, and hallucination-free final response.

4. **Concrete application example:**
   A highly practical application is an **Internal Legal Compliance Search Assistant** for a financial institution. Since legal terminology relies heavily on subtle, contextual relationships between words, keyword searches often fail. By using BERT-based embeddings to index corporate compliance policy manuals, the system can understand conceptual queries (e.g., "Can I accept gifts from vendors?") and accurately retrieve the exact clauses and limitations, even if the query uses completely different wording from the official policy documentation.